# Multi-Agent Sepsis Prediction — Ablation Study

Systematic parameter ablation on **full MIMIC-IV (65,297 patients)**. Each experiment changes **one parameter** from the baseline to measure its individual impact.

| ID | Parameter Changed | Value | Baseline |
|----|-------------------|-------|----------|
| E1 | — (Baseline) | — | 64/2, LR=1e-4, drop=0.3, alpha=0.25, gamma=2.0, seq=24, wd=1e-4 |
| E2 | Model size | 32h/1L | 64h/2L |
| E3 | Learning rate | 5e-4 | 1e-4 |
| E4 | Dropout | 0.4 | 0.3 |
| E5 | Focal alpha | 0.35 | 0.25 |
| E6 | Focal gamma | 1.0 | 2.0 |
| E7 | Focal gamma | 3.0 | 2.0 |
| E8 | Sequence length | 12h | 24h |
| E9 | Sequence length | 48h | 24h |
| E10 | Weight decay | 1e-3 | 1e-4 |

## Key Features:
- **Checkpoint/resume per experiment**: Saves after every epoch. If Colab disconnects, re-run and it resumes.
- **Dataset cache**: Sequences saved to Drive so they don't need rebuilding after disconnect.
- **Speed optimized**: BS=1024, AMP, torch.compile, cudnn.benchmark, patience=5

**Author:** Jason | **Date:** March 2026

## 0. Keep Colab Alive

In [ ]:
import IPython
display(IPython.display.Javascript('''
function KeepAlive() {
    console.log("Keeping alive: " + new Date().toLocaleTimeString());
    google.colab.kernel.invokeFunction("server.ping", [], {});
}
setInterval(KeepAlive, 60000)
'''))
print('Keep-alive active â€” pings every 60s')

## 1. Setup

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm h5py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix
)

PROJECT_PATH = '/content/drive/MyDrive/Sepsis'
DATA_PATH = f'{PROJECT_PATH}/data/processed/mimic_harmonized'
MODEL_PATH = f'{PROJECT_PATH}/models'

# Always pull fresh code from GitHub (Drive's src/ is unreliable due to sync delays)
CODE_PATH = '/content/sepsis-code'
GITHUB_REPO = 'https://github.com/Kai-clou/sepsis-prediction.git'
import subprocess
if os.path.exists(CODE_PATH):
    subprocess.run(['git', '-C', CODE_PATH, 'pull', '--quiet'], check=True)
    print(f'Updated existing code at {CODE_PATH}')
else:
    subprocess.run(['git', 'clone', '--quiet', GITHUB_REPO, CODE_PATH], check=True)
    print(f'Cloned fresh code to {CODE_PATH}')
sys.path.insert(0, f'{CODE_PATH}/src')
os.makedirs(MODEL_PATH, exist_ok=True)

# Verify the model has disabled_agents support (E11/E12/E13)
with open(f'{CODE_PATH}/src/models/multi_agent.py') as _f:
    assert 'disabled_agents' in _f.read(), 'multi_agent.py missing disabled_agents — git pull failed'
print('Model code verified: disabled_agents support present')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from models.multi_agent import MultiAgentSepsisPredictor, FocalLoss, count_parameters
print('Model imported successfully!')

## 2. Experiment Configurations

In [ ]:
VITALS_FEATURES = ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat']
LABS_FEATURES = ['bun', 'chloride', 'creatinine', 'wbc', 'bicarbonate', 'platelets',
                 'magnesium', 'calcium', 'potassium', 'sodium', 'glucose',
                 'fio2', 'ph', 'paco2', 'pao2', 'lactate', 'bilirubin']
ALL_FEATURES = VITALS_FEATURES + LABS_FEATURES

# ============================================================
# ABLATION STUDY: One parameter changed at a time from baseline
# All on full MIMIC-IV (65K patients)
# Baseline: LR=1e-4, 64/2, drop=0.3, alpha=0.25, gamma=2.0, seq=24, wd=1e-4
# ============================================================

BASELINE = {
    'data_file': 'mimic_processed_full.h5',
    'num_patients': None,
    'learning_rate': 1e-4,
    'hidden_dim': 64,
    'num_layers': 2,
    'dropout': 0.3,
    'focal_alpha': 0.25,
    'batch_size': 1024,
}

EXPERIMENTS = {
    'E1': {**BASELINE,
        'description': 'BASELINE (64/2, all defaults)',
    },
    'E2': {**BASELINE,
        'description': 'Ablation: Model size -> 32/1',
        'hidden_dim': 32,
        'num_layers': 1,
    },
    'E3': {**BASELINE,
        'description': 'Ablation: Learning rate -> 5e-4',
        'learning_rate': 5e-4,
    },
    'E4': {**BASELINE,
        'description': 'Ablation: Dropout -> 0.4',
        'dropout': 0.4,
    },
    'E5': {**BASELINE,
        'description': 'Ablation: Focal alpha -> 0.35',
        'focal_alpha': 0.35,
    },
    'E6': {**BASELINE,
        'description': 'Ablation: Focal gamma -> 1.0',
        'focal_gamma': 1.0,
    },
    'E7': {**BASELINE,
        'description': 'Ablation: Focal gamma -> 3.0',
        'focal_gamma': 3.0,
    },
    'E8': {**BASELINE,
        'description': 'Ablation: Sequence length -> 12',
        'sequence_length': 12,
    },
    'E9': {**BASELINE,
        'description': 'Ablation: Sequence length -> 48',
        'sequence_length': 48,
    },
    'E10': {**BASELINE,
        'description': 'Ablation: Weight decay -> 1e-3',
        'weight_decay': 1e-3,
    },
    'E11': {**BASELINE,
        'description': 'Per-agent ablation: drop Vitals Agent',
        'hidden_dim': 32, 'num_layers': 1,
        'disabled_agents': ['vitals'],
    },
    'E12': {**BASELINE,
        'description': 'Per-agent ablation: drop Labs Agent',
        'hidden_dim': 32, 'num_layers': 1,
        'disabled_agents': ['labs'],
    },
    'E13': {**BASELINE,
        'description': 'Per-agent ablation: drop Trend Agent',
        'hidden_dim': 32, 'num_layers': 1,
        'disabled_agents': ['trend'],
    },
}

SHARED = {
    'sequence_length': 24,
    'batch_size': 1024,
    'weight_decay': 1e-4,
    'epochs': 30,
    'patience': 5,
    'focal_gamma': 2.0,
    'random_seed': 42,
    'test_size': 0.2,
    'val_size': 0.1,
}

print(f'Defined {len(EXPERIMENTS)} ablation experiments:')
for name, cfg in EXPERIMENTS.items():
    bs = cfg.get('batch_size', SHARED['batch_size'])
    seq = cfg.get('sequence_length', SHARED['sequence_length'])
    fg = cfg.get('focal_gamma', SHARED['focal_gamma'])
    wd = cfg.get('weight_decay', SHARED['weight_decay'])
    desc = cfg["description"]
    h, l = cfg["hidden_dim"], cfg["num_layers"]
    lr, dr, fa = cfg["learning_rate"], cfg["dropout"], cfg["focal_alpha"]
    print(f"  {name}: {desc}  [H={h}, L={l}, LR={lr}, drop={dr}, a={fa}, g={fg}, seq={seq}, wd={wd}, BS={bs}]")

## 3. Dataset Class

In [ ]:
class SepsisSequenceDataset(Dataset):
    """Creates 24-hour sliding window sequences. Supports memory-mapped arrays."""

    def __init__(self, sequences, labels, missing_mask, n_vitals):
        self.sequences = sequences
        self.labels = labels
        self.missing_mask = missing_mask
        self.n_vitals = n_vitals

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = np.array(self.sequences[idx])  # single copy from mmap
        mask = np.array(self.missing_mask[idx])
        return {
            'vitals': torch.from_numpy(seq[:, :self.n_vitals]),
            'labs': torch.from_numpy(seq[:, self.n_vitals:].copy()),
            'labs_mask': torch.from_numpy(mask[:, self.n_vitals:].copy()),
            'all_features': torch.from_numpy(seq),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32).unsqueeze(0),
        }


def build_sequences_chunked(df, vitals_features, labs_features, seq_length=24,
                            chunk_size=2000, save_path=None):
    """
    Build sequences from a DataFrame in chunks to avoid OOM.
    Uses memory-mapped numpy files for large datasets.
    """
    all_features = vitals_features + labs_features
    n_features = len(all_features)

    # Fast count using groupby (not per-patient loop)
    print(f'    Counting sequences...')
    patient_counts = df.groupby('subject_id').size()
    eligible = patient_counts[patient_counts >= seq_length]
    total_seqs = int((eligible - seq_length + 1).sum())
    patient_ids = eligible.index.values
    n_patients = len(patient_ids)
    print(f'    {n_patients:,} eligible patients, {total_seqs:,} total sequences')

    if total_seqs == 0:
        return np.array([]), np.array([]), np.array([])

    # Allocate arrays (memory-mapped if save_path provided and large)
    use_mmap = save_path is not None and total_seqs > 100000
    if use_mmap:
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        seq_file = save_path + '_seqs.npy'
        lbl_file = save_path + '_labels.npy'
        mask_file = save_path + '_masks.npy'
        sequences = np.lib.format.open_memmap(
            seq_file, mode='w+', dtype=np.float32,
            shape=(total_seqs, seq_length, n_features))
        labels = np.lib.format.open_memmap(
            lbl_file, mode='w+', dtype=np.float32, shape=(total_seqs,))
        missing_mask = np.lib.format.open_memmap(
            mask_file, mode='w+', dtype=np.float32,
            shape=(total_seqs, seq_length, n_features))
        print(f'    Using memory-mapped files at {save_path}*')
    else:
        sequences = np.empty((total_seqs, seq_length, n_features), dtype=np.float32)
        labels = np.empty((total_seqs,), dtype=np.float32)
        missing_mask = np.empty((total_seqs, seq_length, n_features), dtype=np.float32)

    # Build in chunks using groupby (much faster than per-patient filtering)
    idx = 0
    n_chunks = (n_patients + chunk_size - 1) // chunk_size

    for chunk_i in range(n_chunks):
        chunk_start = chunk_i * chunk_size
        chunk_end = min(chunk_start + chunk_size, n_patients)
        chunk_pids = set(patient_ids[chunk_start:chunk_end])

        chunk_df = df[df['subject_id'].isin(chunk_pids)]
        grouped = chunk_df.groupby('subject_id')

        for pid, group in tqdm(grouped, desc=f'Chunk {chunk_i+1}/{n_chunks}',
                               total=len(chunk_pids), leave=False):
            group = group.sort_values('charttime')
            if len(group) < seq_length:
                continue

            values = group[all_features].values.astype(np.float32)
            sep_labels = group['sepsis_label'].values.astype(np.float32)
            is_nan = np.isnan(values).astype(np.float32)
            values_clean = np.nan_to_num(values, nan=0.0)

            n_seqs = len(group) - seq_length + 1
            for i in range(n_seqs):
                sequences[idx] = values_clean[i:i + seq_length]
                missing_mask[idx] = is_nan[i:i + seq_length]
                labels[idx] = sep_labels[i + seq_length - 1]
                idx += 1

        del chunk_df, grouped
        gc.collect()

    positive_rate = labels[:idx].mean() * 100
    print(f'    -> {idx:,} sequences, {positive_rate:.1f}% positive')

    if use_mmap:
        sequences.flush()
        labels.flush()
        missing_mask.flush()

    return sequences[:idx], labels[:idx], missing_mask[:idx]

## 4. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    use_amp = scaler is not None

    for batch in tqdm(loader, desc='Train', leave=False):
        vitals = batch['vitals'].to(device, non_blocking=True)
        labs = batch['labs'].to(device, non_blocking=True)
        labs_mask = batch['labs_mask'].to(device, non_blocking=True)
        all_features = batch['all_features'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            output = model(vitals, labs, labs_mask, all_features)
            loss = criterion(output['logits'], labels)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        all_preds.extend(output['probability'].detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    return (
        total_loss / len(loader),
        roc_auc_score(all_labels, all_preds),
        average_precision_score(all_labels, all_preds),
    )


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_weights = [], [], []

    with torch.no_grad(), torch.cuda.amp.autocast():
        for batch in tqdm(loader, desc='Eval', leave=False):
            vitals = batch['vitals'].to(device, non_blocking=True)
            labs = batch['labs'].to(device, non_blocking=True)
            labs_mask = batch['labs_mask'].to(device, non_blocking=True)
            all_features = batch['all_features'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)

            output = model(vitals, labs, labs_mask, all_features)
            loss = criterion(output['logits'], labels)

            total_loss += loss.item()
            all_preds.extend(output['probability'].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_weights.extend(output['agent_weights'].cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    return (
        total_loss / len(loader),
        roc_auc_score(all_labels, all_preds),
        average_precision_score(all_labels, all_preds),
        all_preds,
        all_labels,
        np.array(all_weights),
    )

## 5. Data Preparation Helper

In [ ]:
def prepare_data(data_file, num_patients, vitals_features, labs_features, seed=42):
    """
    Load data, normalize, split. Returns train/val/test DataFrames and feature stats.
    For large datasets, computes stats before loading full data to manage memory.
    """
    all_features = vitals_features + labs_features
    
    print(f'  Loading {data_file}...')
    df = pd.read_hdf(f'{DATA_PATH}/{data_file}')
    print(f'  Loaded: {len(df):,} rows, {df["subject_id"].nunique():,} patients')

    # Subsample if needed (v1)
    if num_patients is not None:
        all_patient_ids = df['subject_id'].unique()
        if num_patients < len(all_patient_ids):
            patient_labels_all = df.groupby('subject_id')['sepsis_label'].max()
            selected_ids, _ = train_test_split(
                all_patient_ids, train_size=num_patients,
                stratify=patient_labels_all.loc[all_patient_ids], random_state=seed
            )
            df = df[df['subject_id'].isin(selected_ids)].copy()
            print(f'  Subsampled to {len(selected_ids)} patients')

    # Stratified split BEFORE normalization (split on patient IDs, not data)
    patient_ids = df['subject_id'].unique()
    patient_labels = df.groupby('subject_id')['sepsis_label'].max().loc[patient_ids]

    train_val_ids, test_ids = train_test_split(
        patient_ids, test_size=SHARED['test_size'],
        stratify=patient_labels, random_state=seed
    )
    train_val_labels = patient_labels.loc[train_val_ids]
    train_ids, val_ids = train_test_split(
        train_val_ids,
        test_size=SHARED['val_size'] / (1 - SHARED['test_size']),
        stratify=train_val_labels, random_state=seed
    )

    print(f'  Split: Train={len(train_ids)} pts, Val={len(val_ids)} pts, Test={len(test_ids)} pts')

    # Normalize using FULL dataset stats (before splitting into separate DFs)
    feature_stats = {}
    for feature in all_features:
        if feature not in df.columns:
            continue
        mean = df[feature].mean()
        std = df[feature].std()
        if std == 0 or pd.isna(std):
            std = 1.0
        feature_stats[feature] = {'mean': float(mean), 'std': float(std)}
        df[feature] = (df[feature] - mean) / std

    # Split into separate DataFrames
    train_df = df[df['subject_id'].isin(train_ids)]
    val_df = df[df['subject_id'].isin(val_ids)]
    test_df = df[df['subject_id'].isin(test_ids)]

    print(f'  Train={len(train_df):,} obs, Val={len(val_df):,} obs, Test={len(test_df):,} obs')

    return train_df, val_df, test_df, feature_stats

## 6. Run One Experiment (with Checkpoint/Resume)

Each experiment:
1. Checks if already completed (results.json exists) â†’ **skip**
2. Checks for dataset cache â†’ **reload** if exists, build if not
3. Checks for training checkpoint â†’ **resume** from last epoch if exists
4. Saves checkpoint every epoch to Drive

In [ ]:
def copy_to_local_ssd(mmap_dir, local_cache_dir):
    """
    Copy mmap files from Drive to local SSD for fast training.
    Returns True if successful, False if failed (will fall back to Drive).
    """
    import shutil

    # Check available disk space
    stat = os.statvfs('/content')
    free_gb = (stat.f_bavail * stat.f_frsize) / 1e9

    # Estimate needed space
    total_size = 0
    for split in ['train', 'val', 'test']:
        for suffix in ['_seqs.npy', '_labels.npy', '_masks.npy']:
            f = f'{mmap_dir}/{split}{suffix}'
            if os.path.exists(f):
                total_size += os.path.getsize(f)
    needed_gb = total_size / 1e9

    print(f'    Local disk: {free_gb:.1f} GB free, need {needed_gb:.1f} GB')

    if needed_gb > free_gb * 0.8:  # Leave 20% buffer
        print(f'    WARNING: Not enough local disk space. Will train from Drive (slower).')
        return False

    try:
        os.makedirs(local_cache_dir, exist_ok=True)
        for split in ['train', 'val', 'test']:
            for suffix in ['_seqs.npy', '_labels.npy', '_masks.npy']:
                src = f'{mmap_dir}/{split}{suffix}'
                dst = f'{local_cache_dir}/{split}{suffix}'
                if not os.path.exists(dst):
                    print(f'    Copying {split}{suffix}...', end=' ', flush=True)
                    shutil.copy2(src, dst)
                    size_gb = os.path.getsize(dst) / 1e9
                    print(f'done ({size_gb:.1f} GB)')
                else:
                    print(f'    {split}{suffix} already on local SSD')
        return True
    except Exception as e:
        print(f'    WARNING: Copy failed ({e}). Will train from Drive (slower).')
        # Clean up partial copies
        if os.path.exists(local_cache_dir):
            shutil.rmtree(local_cache_dir, ignore_errors=True)
        return False


def run_experiment(version, config):
    """
    Run one experiment with full checkpoint/resume support.
    Uses AMP (mixed precision) for ~2x speedup on GPU.
    Safety: disk space check, fallback to Drive, per-epoch checkpoints.
    """
    save_dir = f'{MODEL_PATH}/{version}'
    checkpoint_dir = f'{save_dir}/checkpoints'
    mmap_dir = f'{save_dir}/mmap_cache'
    local_cache_dir = f'/content/cache/{version}'
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Per-version batch size (v7 uses 512, others use 32)
    batch_size = config.get('batch_size', SHARED['batch_size'])

    # Per-experiment overrides for ablation study
    seq_length = config.get('sequence_length', SHARED['sequence_length'])
    focal_gamma = config.get('focal_gamma', SHARED['focal_gamma'])
    weight_decay = config.get('weight_decay', SHARED['weight_decay'])
    epochs = config.get('epochs', SHARED['epochs'])
    patience = config.get('patience', SHARED['patience'])

    # ---- SKIP if already completed ----
    if os.path.exists(f'{save_dir}/results.json'):
        with open(f'{save_dir}/results.json') as f:
            r = json.load(f)
        print(f'\n{"="*70}')
        print(f'  {version}: ALREADY COMPLETE â€” AUROC={r["test_auroc"]:.4f}, AUPRC={r["test_auprc"]:.4f}')
        print(f'{"="*70}')
        return r

    print(f'\n{"="*70}')
    print(f'  EXPERIMENT {version}: {config["description"]}')
    print(f'  Data: {config["data_file"]}, LR={config["learning_rate"]}, '
          f'Hidden={config["hidden_dim"]}, Layers={config["num_layers"]}, '
          f'Dropout={config["dropout"]}, Focal_a={config["focal_alpha"]}, BS={batch_size}')
    print(f'{"="*70}')

    start_time = time.time()

    # ---- Check available features ----
    df_sample = pd.read_hdf(f'{DATA_PATH}/{config["data_file"]}', stop=10)
    vitals_avail = [f for f in VITALS_FEATURES if f in df_sample.columns]
    labs_avail = [f for f in LABS_FEATURES if f in df_sample.columns]
    all_avail = vitals_avail + labs_avail
    n_vitals = len(vitals_avail)
    del df_sample

    # ---- Determine if this is a large dataset ----
    is_large = config['data_file'] == 'mimic_processed_full.h5'

    # ---- DATASET: load cache or build ----
    cache_file = f'{save_dir}/dataset_cache.npz'

    # Check ALL 9 mmap files exist (incomplete cache from a crashed build must trigger rebuild)
    mmap_ready = all(
        os.path.exists(f'{mmap_dir}/{split}{suffix}')
        for split in ['train', 'val', 'test']
        for suffix in ['_seqs.npy', '_labels.npy', '_masks.npy']
    ) and os.path.exists(f'{save_dir}/feature_stats.json')

    if is_large and not mmap_ready:
        # Clean up incomplete mmap cache before rebuilding
        import shutil
        if os.path.exists(mmap_dir):
            print(f'\n[DATA] Incomplete mmap cache detected â€” deleting {mmap_dir}/ and rebuilding...')
            shutil.rmtree(mmap_dir, ignore_errors=True)
        # Also delete stale checkpoints/best_model from previous incomplete runs
        if os.path.exists(checkpoint_dir):
            shutil.rmtree(checkpoint_dir, ignore_errors=True)
            os.makedirs(checkpoint_dir, exist_ok=True)
            print(f'  Deleted stale checkpoints from previous incomplete run')
        stale_model = f'{save_dir}/best_model.pt'
        if os.path.exists(stale_model):
            os.remove(stale_model)
            print(f'  Deleted stale best_model.pt from previous incomplete run')

    # Track where data lives for DataLoader config
    using_local_ssd = False

    if is_large and mmap_ready:
        # Try to copy to local SSD for speed; fall back to Drive if needed
        print('\n[DATA] Preparing cached data for training...')
        using_local_ssd = copy_to_local_ssd(mmap_dir, local_cache_dir)
        data_dir = local_cache_dir if using_local_ssd else mmap_dir

        print(f'\n[DATA] Loading from {"local SSD" if using_local_ssd else "Drive (slower)"}...')
        train_seqs = np.load(f'{data_dir}/train_seqs.npy', mmap_mode='r')
        train_labels = np.load(f'{data_dir}/train_labels.npy', mmap_mode='r')
        train_masks = np.load(f'{data_dir}/train_masks.npy', mmap_mode='r')
        val_seqs = np.load(f'{data_dir}/val_seqs.npy', mmap_mode='r')
        val_labels = np.load(f'{data_dir}/val_labels.npy', mmap_mode='r')
        val_masks = np.load(f'{data_dir}/val_masks.npy', mmap_mode='r')
        test_seqs = np.load(f'{data_dir}/test_seqs.npy', mmap_mode='r')
        test_labels = np.load(f'{data_dir}/test_labels.npy', mmap_mode='r')
        test_masks = np.load(f'{data_dir}/test_masks.npy', mmap_mode='r')
        with open(f'{save_dir}/feature_stats.json') as f:
            feature_stats = json.load(f)
        print(f'  Loaded: Train={len(train_seqs):,}, Val={len(val_seqs):,}, Test={len(test_seqs):,}')

    elif not is_large and os.path.exists(cache_file):
        print('\n[DATA] Loading cached datasets...')
        cache = np.load(cache_file, allow_pickle=True)
        train_seqs = cache['train_seqs']
        train_labels = cache['train_labels']
        train_masks = cache['train_masks']
        val_seqs = cache['val_seqs']
        val_labels = cache['val_labels']
        val_masks = cache['val_masks']
        test_seqs = cache['test_seqs']
        test_labels = cache['test_labels']
        test_masks = cache['test_masks']
        with open(f'{save_dir}/feature_stats.json') as f:
            feature_stats = json.load(f)
        print(f'  Loaded: Train={len(train_seqs):,}, Val={len(val_seqs):,}, Test={len(test_seqs):,}')

    else:
        # Build from scratch
        print('\n[DATA] Preparing data...')
        train_df, val_df, test_df, feature_stats = prepare_data(
            config['data_file'], config['num_patients'], vitals_avail, labs_avail
        )
        with open(f'{save_dir}/feature_stats.json', 'w') as f:
            json.dump(feature_stats, f, indent=2)

        if is_large:
            os.makedirs(mmap_dir, exist_ok=True)
            print('\n[DATA] Building sequences (chunked, memory-mapped)...')
            print('  Train:')
            train_seqs, train_labels, train_masks = build_sequences_chunked(
                train_df, vitals_avail, labs_avail, seq_length,
                chunk_size=2000, save_path=f'{mmap_dir}/train')
            del train_df; gc.collect()

            print('  Val:')
            val_seqs, val_labels, val_masks = build_sequences_chunked(
                val_df, vitals_avail, labs_avail, seq_length,
                chunk_size=2000, save_path=f'{mmap_dir}/val')
            del val_df; gc.collect()

            print('  Test:')
            test_seqs, test_labels, test_masks = build_sequences_chunked(
                test_df, vitals_avail, labs_avail, seq_length,
                chunk_size=2000, save_path=f'{mmap_dir}/test')
            del test_df; gc.collect()

            print(f'\n[DATA] Memory-mapped cache saved to {mmap_dir}/')

            # Try to copy to local SSD
            print('\n[DATA] Copying to local SSD for fast training...')
            using_local_ssd = copy_to_local_ssd(mmap_dir, local_cache_dir)

            if using_local_ssd:
                # Reload from local SSD
                train_seqs = np.load(f'{local_cache_dir}/train_seqs.npy', mmap_mode='r')
                train_labels = np.load(f'{local_cache_dir}/train_labels.npy', mmap_mode='r')
                train_masks = np.load(f'{local_cache_dir}/train_masks.npy', mmap_mode='r')
                val_seqs = np.load(f'{local_cache_dir}/val_seqs.npy', mmap_mode='r')
                val_labels = np.load(f'{local_cache_dir}/val_labels.npy', mmap_mode='r')
                val_masks = np.load(f'{local_cache_dir}/val_masks.npy', mmap_mode='r')
                test_seqs = np.load(f'{local_cache_dir}/test_seqs.npy', mmap_mode='r')
                test_labels = np.load(f'{local_cache_dir}/test_labels.npy', mmap_mode='r')
                test_masks = np.load(f'{local_cache_dir}/test_masks.npy', mmap_mode='r')
            # else: keep using the mmap arrays already in memory from build step
        else:
            print('\n[DATA] Building sequences...')
            print('  Train:')
            train_seqs, train_labels, train_masks = build_sequences_chunked(
                train_df, vitals_avail, labs_avail, seq_length,
                chunk_size=5000)
            print('  Val:')
            val_seqs, val_labels, val_masks = build_sequences_chunked(
                val_df, vitals_avail, labs_avail, seq_length,
                chunk_size=5000)
            print('  Test:')
            test_seqs, test_labels, test_masks = build_sequences_chunked(
                test_df, vitals_avail, labs_avail, seq_length,
                chunk_size=5000)

            del train_df, val_df, test_df
            gc.collect()

            print('\n[DATA] Saving cache to Drive...')
            np.savez(cache_file,
                     train_seqs=train_seqs, train_labels=train_labels, train_masks=train_masks,
                     val_seqs=val_seqs, val_labels=val_labels, val_masks=val_masks,
                     test_seqs=test_seqs, test_labels=test_labels, test_masks=test_masks)
            print(f'  Cache saved ({os.path.getsize(cache_file)/1e6:.0f} MB)')

    # ---- Create DataLoaders ----
    # More workers + persistent + prefetch for pipelining I/O with GPU compute
    n_workers = 4 if (using_local_ssd or not is_large) else 0
    use_persistent = n_workers > 0
    print(f'\n[DATA] DataLoader: batch_size={batch_size}, workers={n_workers}, '
          f'persistent={use_persistent}, '
          f'source={"local SSD" if using_local_ssd else ("RAM" if not is_large else "Drive")}')

    train_loader = DataLoader(
        SepsisSequenceDataset(train_seqs, train_labels, train_masks, n_vitals),
        batch_size=batch_size, shuffle=True, num_workers=n_workers,
        pin_memory=True, persistent_workers=use_persistent,
        prefetch_factor=4 if n_workers > 0 else None)
    val_loader = DataLoader(
        SepsisSequenceDataset(val_seqs, val_labels, val_masks, n_vitals),
        batch_size=batch_size, shuffle=False, num_workers=n_workers,
        pin_memory=True, persistent_workers=use_persistent,
        prefetch_factor=4 if n_workers > 0 else None)
    test_loader = DataLoader(
        SepsisSequenceDataset(test_seqs, test_labels, test_masks, n_vitals),
        batch_size=batch_size, shuffle=False, num_workers=n_workers,
        pin_memory=True, persistent_workers=use_persistent,
        prefetch_factor=4 if n_workers > 0 else None)

    # ---- Initialize model ----
    print(f'\n[MODEL] Initializing...')
    model = MultiAgentSepsisPredictor(
        vitals_dim=n_vitals,
        labs_dim=len(labs_avail),
        all_features_dim=len(all_avail),
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        dropout=config['dropout'],
        disabled_agents=config.get('disabled_agents', None)
    ).to(device)

    criterion = FocalLoss(alpha=config['focal_alpha'], gamma=focal_gamma)
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # AMP (mixed precision) â€” ~2x speedup on T4/A100
    use_amp = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    if use_amp:
        print(f'  AMP (mixed precision): ENABLED')

    n_params = count_parameters(model)
    print(f'  Parameters: {n_params:,}')

    # ---- RESUME from checkpoint if exists ----
    ckpt_file = f'{checkpoint_dir}/latest.pt'
    hist_file = f'{checkpoint_dir}/history.json'
    start_epoch = 0
    best_val_auroc = 0
    patience_counter = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_auroc': [], 'val_auroc': [],
        'train_auprc': [], 'val_auprc': [],
    }

    if os.path.exists(ckpt_file):
        print(f'\n[RESUME] Loading checkpoint...')
        ckpt = torch.load(ckpt_file, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_auroc = ckpt['best_val_auroc']
        patience_counter = ckpt['patience_counter']
        if os.path.exists(hist_file):
            with open(hist_file) as f:
                history = json.load(f)
        print(f'  Resumed from epoch {start_epoch} (best AUROC: {best_val_auroc:.4f}, patience: {patience_counter})')
    else:
        print(f'\n[TRAIN] Starting fresh')

    # ---- Training loop ----
    print(f'[TRAIN] Epochs {start_epoch+1} to {epochs} (patience={patience})')
    print(f'  Batches per epoch: ~{len(train_loader):,} train, ~{len(val_loader):,} val\n')

    for epoch in range(start_epoch, epochs):
        epoch_start = time.time()

        train_loss, train_auroc, train_auprc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler=scaler)
        val_loss, val_auroc, val_auprc, _, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_auroc)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_auroc'].append(train_auroc)
        history['val_auroc'].append(val_auroc)
        history['train_auprc'].append(train_auprc)
        history['val_auprc'].append(val_auprc)

        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]['lr']

        print(f'  Epoch {epoch+1:2d}/{epochs} | '
              f'Train: {train_auroc:.4f} | Val: {val_auroc:.4f} | '
              f'AUPRC: {val_auprc:.4f} | LR: {current_lr:.1e} | '
              f'{epoch_time:.0f}s', end='')

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_auroc': val_auroc,
                'config': config,
            }, f'{save_dir}/best_model.pt')
            print(' *BEST*')
        else:
            patience_counter += 1
            print(f' (patience {patience_counter}/{patience})')

        # Save checkpoint every epoch (to Drive for persistence)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_auroc': best_val_auroc,
            'patience_counter': patience_counter,
        }, ckpt_file)
        with open(hist_file, 'w') as f:
            json.dump(history, f)

        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # ---- Test evaluation ----
    print(f'\n[TEST] Evaluating best model...')
    best_ckpt = torch.load(f'{save_dir}/best_model.pt', weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])

    test_loss, test_auroc, test_auprc, test_preds, test_labels_arr, agent_weights = evaluate(
        model, test_loader, criterion, device
    )

    prec, rec, thresholds_pr = precision_recall_curve(test_labels_arr, test_preds)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = float(thresholds_pr[optimal_idx]) if optimal_idx < len(thresholds_pr) else 0.5
    best_f1 = float(f1_scores[optimal_idx])

    test_preds_binary = (test_preds >= optimal_threshold).astype(int)
    cm = confusion_matrix(test_labels_arr, test_preds_binary)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    avg_weights = agent_weights.mean(axis=0)
    sepsis_mask = test_labels_arr == 1
    sepsis_weights = agent_weights[sepsis_mask].mean(axis=0) if sepsis_mask.sum() > 0 else avg_weights
    nonsepsis_weights = agent_weights[~sepsis_mask].mean(axis=0) if (~sepsis_mask).sum() > 0 else avg_weights

    elapsed = time.time() - start_time

    print(f'\n  {"="*50}')
    print(f'  {version} TEST RESULTS')
    print(f'  {"="*50}')
    print(f'    AUROC:       {test_auroc:.4f}')
    print(f'    AUPRC:       {test_auprc:.4f}')
    print(f'    F1:          {best_f1:.4f} (threshold={optimal_threshold:.3f})')
    print(f'    Sensitivity: {sensitivity:.4f}')
    print(f'    Specificity: {specificity:.4f}')
    print(f'    PPV:         {ppv:.4f}')
    print(f'    NPV:         {npv:.4f}')
    print(f'    Agents:      Vitals={avg_weights[0]:.1%} Labs={avg_weights[1]:.1%} Trend={avg_weights[2]:.1%}')
    print(f'    Time:        {elapsed/60:.1f} min')

    # ---- Save results ----
    results = {
        'version': version,
        'description': config['description'],
        'test_auroc': float(test_auroc),
        'test_auprc': float(test_auprc),
        'test_loss': float(test_loss),
        'f1': float(best_f1),
        'optimal_threshold': optimal_threshold,
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'ppv': float(ppv),
        'npv': float(npv),
        'confusion_matrix': cm.tolist(),
        'best_epoch': int(best_ckpt['epoch']),
        'n_params': n_params,
        'training_time_min': round(elapsed / 60, 1),
        'agent_weights_overall': [float(w) for w in avg_weights],
        'agent_weights_sepsis': [float(w) for w in sepsis_weights],
        'agent_weights_nonsepsis': [float(w) for w in nonsepsis_weights],
        'history': history,
        'config': {**config, **SHARED, 'batch_size': batch_size},
        'timestamp': datetime.now().isoformat(),
    }

    with open(f'{save_dir}/results.json', 'w') as f:
        json.dump(results, f, indent=2)

    np.savez(f'{save_dir}/test_predictions.npz',
             preds=test_preds, labels=test_labels_arr, agent_weights=agent_weights)

    # Training curves plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title(f'{version} Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['train_auroc'], label='Train'); axes[1].plot(history['val_auroc'], label='Val')
    axes[1].set_title(f'{version} AUROC'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].plot(history['train_auprc'], label='Train'); axes[2].plot(history['val_auprc'], label='Val')
    axes[2].set_title(f'{version} AUPRC'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'  Saved to {save_dir}/')

    # Cleanup
    del model, optimizer, scheduler, criterion
    del train_loader, val_loader, test_loader
    if not is_large:
        del train_seqs, train_labels, train_masks
        del val_seqs, val_labels, val_masks
        del test_seqs, test_labels, test_masks
    torch.cuda.empty_cache()
    gc.collect()

    return results

## 7. RUN ALL 7 EXPERIMENTS

**If Colab disconnects:** Just re-run from this cell. It will:
1. Skip completed versions (results.json exists)
2. Reload cached datasets (dataset_cache.npz exists)
3. Resume training from last epoch (latest.pt checkpoint)

In [ ]:
# ============================================================
# ABLATION STUDY: Run all 10 experiments
# E1 and E2 auto-skip if results.json exists (reused from v7 runs)
# E3-E10 are new experiments, ~25-30 min each on A100
# ============================================================
VERSIONS_TO_RUN = [
    'E1',   # Baseline (64/2, all defaults)
    'E2',   # Model size -> 32/1
    'E3',   # Learning rate -> 5e-4
    'E4',   # Dropout -> 0.4
    'E5',   # Focal alpha -> 0.35
    'E6',   # Focal gamma -> 1.0
    'E7',   # Focal gamma -> 3.0
    'E8',   # Sequence length -> 12
    'E9',   # Sequence length -> 48
    'E10',  # Weight decay -> 1e-3
    'E11',  # Per-agent: drop Vitals
    'E12',  # Per-agent: drop Labs
    'E13',  # Per-agent: drop Trend
]

all_results = {}
total_start = time.time()

for version in VERSIONS_TO_RUN:
    config = EXPERIMENTS[version]
    results = run_experiment(version, config)
    all_results[version] = results

total_time = (time.time() - total_start) / 60
print()
print("=" * 70)
print(f"ALL {len(VERSIONS_TO_RUN)} EXPERIMENTS COMPLETE in {total_time:.0f} minutes")
print("=" * 70)

## 8. Load & Compare All Ablation Results

In [ ]:
# Load all ablation results
all_results = {}
ablation_ids = ['E1','E2','E3','E4','E5','E6','E7','E8','E9','E10']

for eid in ablation_ids:
    path = f'{MODEL_PATH}/{eid}/results.json'
    if os.path.exists(path):
        with open(path) as f:
            all_results[eid] = json.load(f)

print(f'Loaded results for: {list(all_results.keys())}')

# E2 is the best overall model (best AUROC+AUPRC+F1 balance)
# E8 has highest AUROC (0.7704) but worst AUPRC/F1
best_overall = 'E2'
best_auroc_e = max(all_results, key=lambda e: all_results[e]['test_auroc'])
worst_auroc_e = min(all_results, key=lambda e: all_results[e]['test_auroc'])

# Ablation labels for display
ABLATION_LABELS = {
    'E1': 'Baseline (64/2)',
    'E2': 'Model: 32/1',
    'E3': 'LR: 5e-4',
    'E4': 'Dropout: 0.4',
    'E5': 'Alpha: 0.35',
    'E6': 'Gamma: 1.0',
    'E7': 'Gamma: 3.0',
    'E8': 'Seq: 12h',
    'E9': 'Seq: 48h',
    'E10': 'WD: 1e-3',
}

eids = [e for e in ablation_ids if e in all_results]

print(f"Best overall: {best_overall} ({ABLATION_LABELS[best_overall]}) = {all_results[best_overall]['test_auroc']:.4f} AUROC, {all_results[best_overall]['test_auprc']:.4f} AUPRC")
print(f"Best AUROC:   {best_auroc_e} ({ABLATION_LABELS[best_auroc_e]}) = {all_results[best_auroc_e]['test_auroc']:.4f}")
print(f"Worst AUROC:  {worst_auroc_e} ({ABLATION_LABELS[worst_auroc_e]}) = {all_results[worst_auroc_e]['test_auroc']:.4f}")

## 9. Ablation Comparison Plots

In [ ]:
aurocs = [all_results[e]['test_auroc'] for e in eids]
auprcs = [all_results[e]['test_auprc'] for e in eids]
xlabels = [ABLATION_LABELS.get(e, e) for e in eids]

colors = []
for e in eids:
    if e == best_overall:
        colors.append('#27ae60')   # Green for best overall (E2)
    elif e == worst_auroc_e:
        colors.append('#e74c3c')   # Red for worst
    elif e == 'E1':
        colors.append('#f39c12')   # Orange for baseline
    else:
        colors.append('#3498db')   # Blue for others

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# AUROC
bars = axes[0].bar(xlabels, aurocs, color=colors)
axes[0].set_ylabel('AUROC')
axes[0].set_title('AUROC by Ablation', fontweight='bold')
axes[0].set_ylim(min(aurocs) - 0.01, max(aurocs) + 0.005)
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, aurocs):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}',
                ha='center', va='bottom', fontsize=8)

# AUPRC
bars = axes[1].bar(xlabels, auprcs, color=colors)
axes[1].set_ylabel('AUPRC')
axes[1].set_title('AUPRC by Ablation', fontweight='bold')
axes[1].set_ylim(min(auprcs) - 0.01, max(auprcs) + 0.005)
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, auprcs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}',
                ha='center', va='bottom', fontsize=8)

# Delta from baseline
baseline_auroc = all_results['E1']['test_auroc']
deltas = [all_results[e]['test_auroc'] - baseline_auroc for e in eids]
delta_colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]
bars = axes[2].bar(xlabels, [d * 100 for d in deltas], color=delta_colors)
axes[2].set_ylabel('Delta AUROC (percentage points)')
axes[2].set_title('Impact vs Baseline', fontweight='bold')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {MODEL_PATH}/ablation_comparison.png')

In [ ]:
# Ablation impact summary (delta from baseline E1)
print('=' * 70)
print('ABLATION IMPACT (delta from E1 baseline)')
print('=' * 70)
b = all_results['E1']
print(f"{'ID':<5} {'Parameter':<22} {'AUROC':>8} {'dAUROC':>8} {'AUPRC':>8} {'dAUPRC':>8} {'F1':>6}")
print('-' * 70)

best_auprc_e = max(all_results, key=lambda x: all_results[x]['test_auprc'])
best_f1_e = max(all_results, key=lambda x: all_results[x]['f1'])

for e in eids:
    r = all_results[e]
    da = r['test_auroc'] - b['test_auroc']
    dp = r['test_auprc'] - b['test_auprc']
    markers = []
    if e == best_overall:
        markers.append('BEST OVERALL')
    if e == best_auroc_e:
        markers.append('best AUROC')
    if e == best_auprc_e:
        markers.append('best AUPRC')
    if e == best_f1_e:
        markers.append('best F1')
    if e == worst_auroc_e:
        markers.append('worst AUROC')
    marker = f' <-- {", ".join(markers)}' if markers else ''
    print(f"{e:<5} {ABLATION_LABELS[e]:<22} {r['test_auroc']:>8.4f} {da:>+8.4f} {r['test_auprc']:>8.4f} {dp:>+8.4f} {r['f1']:>6.4f}{marker}")

## 10. Agent Weights & ROC/PR Curves for Best Model

In [ ]:
# Agent weights comparison across ablations
if len(all_results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    agent_names = ['Vitals', 'Labs', 'Trend']
    agent_colors = ['#1B5E20', '#0D47A1', '#4A148C']

    x = np.arange(len(eids))
    width = 0.25
    for i, (agent, color) in enumerate(zip(agent_names, agent_colors)):
        weights = [all_results[e]['agent_weights_overall'][i] * 100 for e in eids]
        axes[0].bar(x + i * width, weights, width, label=agent, color=color, alpha=0.8)
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels([ABLATION_LABELS[e] for e in eids], rotation=45, ha='right')
    axes[0].set_ylabel('Weight (%)')
    axes[0].set_title('Agent Weights by Ablation', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Sepsis vs Non-Sepsis for best overall model (E2)
    best_r = all_results[best_overall]
    categories = ['Overall', 'Sepsis', 'Non-Sepsis']
    weight_data = [
        best_r['agent_weights_overall'],
        best_r.get('agent_weights_sepsis', best_r['agent_weights_overall']),
        best_r.get('agent_weights_non_sepsis', best_r['agent_weights_overall']),
    ]
    x2 = np.arange(len(categories))
    for i, (agent, color) in enumerate(zip(agent_names, agent_colors)):
        vals = [w[i] * 100 for w in weight_data]
        axes[1].bar(x2 + i * width, vals, width, label=agent, color=color, alpha=0.8)
    axes[1].set_xticks(x2 + width)
    axes[1].set_xticklabels(categories)
    axes[1].set_ylabel('Weight (%)')
    axes[1].set_title(f'Agent Weights - {best_overall} ({ABLATION_LABELS[best_overall]}) [Best Overall]', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{MODEL_PATH}/ablation_agent_weights.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ROC & PR curves for best overall model (E2)
bv = best_overall
npz = np.load(f'{MODEL_PATH}/{bv}/test_predictions.npz')
preds = npz['preds']
labels_arr = npz['labels']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fpr, tpr, _ = roc_curve(labels_arr, preds)
auroc = roc_auc_score(labels_arr, preds)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUROC = {auroc:.4f}')
axes[0].plot([0,1], [0,1], 'k--')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title(f'ROC Curve ({bv} - Best Overall)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

precision, recall, _ = precision_recall_curve(labels_arr, preds)
auprc = average_precision_score(labels_arr, preds)
axes[1].plot(recall, precision, 'g-', linewidth=2, label=f'AUPRC = {auprc:.4f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR Curve ({bv} - Best Overall)', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Probability distribution
preds_flat = preds.flatten()
labels_flat = labels_arr.flatten()
axes[2].hist(preds_flat[labels_flat==0], bins=50, alpha=0.5, label='No Sepsis', color='green')
axes[2].hist(preds_flat[labels_flat==1], bins=50, alpha=0.5, label='Sepsis', color='red')
axes[2].set_xlabel('Predicted Probability'); axes[2].set_ylabel('Count')
axes[2].set_title(f'Prediction Distribution ({bv})', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/ablation_roc_pr_{bv}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves for best overall model (E2) — Figure 4 in thesis
hist = all_results[best_overall]['history']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(hist['train_loss'], 'b-', label='Train', linewidth=1.5)
axes[0].plot(hist['val_loss'], 'r-', label='Validation', linewidth=1.5)
best_epoch = all_results[best_overall]['best_epoch']
axes[0].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUROC
axes[1].plot(hist['train_auroc'], 'b-', label='Train', linewidth=1.5)
axes[1].plot(hist['val_auroc'], 'r-', label='Validation', linewidth=1.5)
axes[1].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUROC')
axes[1].set_title('AUROC Over Training', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUPRC
axes[2].plot(hist['train_auprc'], 'b-', label='Train', linewidth=1.5)
axes[2].plot(hist['val_auprc'], 'r-', label='Validation', linewidth=1.5)
axes[2].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUPRC')
axes[2].set_title('AUPRC Over Training', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Training Curves — {best_overall} ({ABLATION_LABELS[best_overall]})', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {MODEL_PATH}/training_curves.png')

## 11. Final Ablation Summary

In [ ]:
print()
print('=' * 100)
print('FINAL ABLATION SUMMARY')
print('=' * 100)

best_auprc_e = max(all_results, key=lambda e: all_results[e]['test_auprc'])
best_f1_e = max(all_results, key=lambda e: all_results[e]['f1'])

print(f"\n{'ID':<5} {'Ablation':<22} {'AUROC':>7} {'AUPRC':>7} {'F1':>6} {'Sens':>6} {'Spec':>6} {'Time':>6} Status")
print('-' * 100)

for e in eids:
    r = all_results[e]
    t = f"{r.get('training_time_min', 0):.0f}m"
    statuses = []
    if e == best_overall:
        statuses.append('BEST OVERALL')
    if e == best_auroc_e:
        statuses.append('best AUROC')
    if e == best_auprc_e:
        statuses.append('best AUPRC')
    if e == best_f1_e:
        statuses.append('best F1')
    if e == worst_auroc_e:
        statuses.append('worst AUROC')
    status = ', '.join(statuses)
    print(f"{e:<5} {ABLATION_LABELS[e]:<22} {r['test_auroc']:>7.4f} {r['test_auprc']:>7.4f} {r['f1']:>6.4f} {r['sensitivity']:>6.3f} {r['specificity']:>6.3f} {t:>6} {status}")

print()
print('=' * 100)
print(f"Best overall: {best_overall} ({ABLATION_LABELS[best_overall]}) = {all_results[best_overall]['test_auroc']:.4f} AUROC, {all_results[best_overall]['test_auprc']:.4f} AUPRC")
print(f"Best AUROC:   {best_auroc_e} ({ABLATION_LABELS[best_auroc_e]}) = {all_results[best_auroc_e]['test_auroc']:.4f}")
print(f"Best AUPRC:   {best_auprc_e} ({ABLATION_LABELS[best_auprc_e]}) = {all_results[best_auprc_e]['test_auprc']:.4f}")
print(f"Best F1:      {best_f1_e} ({ABLATION_LABELS[best_f1_e]}) = {all_results[best_f1_e]['f1']:.4f}")

print()
print('KEY FINDINGS:')
print('  1. Model size has the largest positive impact (32/1 > 64/2)')
print('  2. Sequence length is the most impactful parameter overall:')
print('     - 12h: best AUROC and specificity, but worst AUPRC/F1')
print('     - 48h: best AUPRC and F1, but worst AUROC')
print('  3. Loss function params (alpha, gamma) and regularization barely matter')
print('  4. Higher LR (5e-4) overfits faster - worse than 1e-4')
print('  5. E2 (32/1) is the best overall model - best balance of AUROC + AUPRC + F1')

## Cleanup (Optional)

Remove mmap caches and checkpoints to save Drive space.

In [ ]:
# Uncomment to clean up caches after successful training:

# import shutil
# for eid in ablation_ids:
#     cache_dir = f'{MODEL_PATH}/{eid}/mmap_cache'
#     ckpt_dir = f'{MODEL_PATH}/{eid}/checkpoints'
#     if os.path.exists(cache_dir):
#         shutil.rmtree(cache_dir)
#         print(f'  Deleted {cache_dir}')
#     if os.path.exists(ckpt_dir):
#         shutil.rmtree(ckpt_dir)
#         print(f'  Deleted {ckpt_dir}')
# print('Cleanup done - best_model.pt and results.json preserved')